# 共通テスト実行ノートブック

ソースノートブックを実行し、対応する pytest テストを HTML レポートで表示する。

**ウィジェット:**
| パラメータ | 説明 |
|---|---|
| `P_BASE_YM` | 基準年月（例: 200512） |
| `P_SOURCE_NOTEBOOK` | ソースコードまたはテストファイルへのパス |

**テストファイルの指定方法:**

| 指定例 | 動作 |
|---|---|
| `../src/import-source-files` | ソースを実行後、`tests/test_import_source_files.py` を自動導出して pytest 実行 |
| `/Workspace/.../tests/test_foo.py` | ソース実行をスキップし、指定テストファイルを直接 pytest 実行 |

> **Note:** テストファイルは Workspace API 経由でエクスポートされるため、サーバーレスでも動作します。

In [0]:
%pip install pytest pytest-html -q

In [0]:
import sys
sys.dont_write_bytecode = True

dbutils.widgets.text("P_BASE_YM", defaultValue="200512", label="基準年月")
dbutils.widgets.text(
    "P_SOURCE_NOTEBOOK",
    defaultValue="../src/",
    label="ソースコードのパス",
)

In [0]:
import os

base_ym = dbutils.widgets.get("P_BASE_YM")
source_notebook = dbutils.widgets.get("P_SOURCE_NOTEBOOK")
source_name = os.path.basename(source_notebook)
source_stem = os.path.splitext(source_name)[0]

# テストファイルが直接指定された場合はソース実行をスキップ
if source_stem.startswith("test_"):
    print(f"テストファイルが指定されているため、ソース実行をスキップ: {source_name}")
else:
    print(f"{source_name} を実行中... (P_BASE_YM={base_ym})")
    dbutils.notebook.run(source_notebook, timeout_seconds=600, arguments={"P_BASE_YM": base_ym})
    print(f"{source_name} 完了")

In [0]:
import os
import sys
import base64
import tempfile
import requests
import pytest

sys.dont_write_bytecode = True

base_ym = dbutils.widgets.get("P_BASE_YM")
source_notebook = dbutils.widgets.get("P_SOURCE_NOTEBOOK")

# ── テストファイルの Workspace API パスを決定 ──
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
_nb_dir = os.path.dirname(ctx.notebookPath().get())
source_name = os.path.basename(source_notebook)
source_stem = os.path.splitext(source_name)[0]

if source_notebook.startswith("/Workspace"):
    test_ws_path = source_notebook.replace("/Workspace", "", 1)
elif source_stem.startswith("test_"):
    test_ws_path = f"{_nb_dir}/tests/{source_stem}.py"
else:
    test_ws_path = f"{_nb_dir}/tests/test_{source_stem.replace('-', '_')}.py"

print(f"テストファイル: {test_ws_path}")

# ── Workspace API 経由でテストファイルをエクスポート ──
token = ctx.apiToken().get()
host = ctx.apiUrl().get()

resp = requests.get(
    f"{host}/api/2.0/workspace/export",
    headers={"Authorization": f"Bearer {token}"},
    params={"path": test_ws_path, "format": "SOURCE"},
)
resp.raise_for_status()
test_source = base64.b64decode(resp.json()["content"]).decode("utf-8")

# Databricks notebook マーカーを除去して純粋な Python ファイルに変換
lines = []
for line in test_source.splitlines():
    s = line.strip()
    if s in ("# Databricks notebook source", "# COMMAND ----------"):
        lines.append("")
    elif s.startswith("# DBTITLE"):
        lines.append("")
    else:
        lines.append(line)
test_source_clean = "\n".join(lines)

tmp_dir = tempfile.mkdtemp()
test_file_name = os.path.basename(test_ws_path)
if not test_file_name.endswith(".py"):
    test_file_name += ".py"
test_file = os.path.join(tmp_dir, test_file_name)
with open(test_file, "w") as f:
    f.write(test_source_clean)
print(f"ローカルテストファイル: {test_file}")

# ── 環境変数設定 ──
os.environ["P_BASE_YM"] = base_ym
os.environ["SOURCE_NOTEBOOK"] = source_notebook

# ── テストモジュールのキャッシュクリア ──
test_module_name = os.path.splitext(os.path.basename(test_file))[0]
for mod_name in list(sys.modules.keys()):
    if test_module_name in mod_name:
        del sys.modules[mod_name]

# ── pytest 実行 ──
html_path = os.path.join(tempfile.mkdtemp(), "report.html")

_exit_code = pytest.main([
    test_file,
    "-v",
    "--tb=short",
    "--no-header",
    "-p", "no:cacheprovider",
    "--import-mode=importlib",
    f"--html={html_path}",
    "--self-contained-html",
])

# ── HTML レポートをノートブック上に表示 ──
with open(html_path, "r") as f:
    displayHTML(f.read())

print(f"pytest exit code: {_exit_code}")

In [0]:
# ── テスト結果の判定（失敗時に例外を発生させる） ──
if _exit_code != 0:
    raise Exception(f"テスト失敗 (exit code: {_exit_code})")
print("✅ 全テスト成功")